# OVMI Paper Experiments

This notebook reproduces the experiments associated with the OVMI paper. To avoid fine-tuning MEG-XL within this repository, we will download the validation and test set predictions from a model already fine-tuned on the LibriBrain brain-to-text dataset.

## Download prediction data from MEG-XL fine-tuned on LibriBrain

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path
from typing import Optional


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "ovmi").exists():
            return candidate
    raise FileNotFoundError("Could not find the OVMI project root from the current directory.")


ensure_package("huggingface_hub")

from huggingface_hub import snapshot_download


DATASET_REPO_ID = "pnpl/ovmi-predictions"
PROJECT_ROOT = find_project_root()
PREDICTIONS_DIR = PROJECT_ROOT / "experiments" / "data" / "ovmi-predictions"

PREDICTIONS_DIR

In [ ]:
snapshot_path = Path(
    snapshot_download(
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        local_dir=PREDICTIONS_DIR,
    )
)

snapshot_path

In [ ]:
seed_dirs = sorted(
    path
    for path in snapshot_path.iterdir()
    if path.is_dir() and not path.name.startswith(".")
)

if len(seed_dirs) != 5:
    raise ValueError(
        f"Expected 5 seed folders in {snapshot_path}, found {len(seed_dirs)}: "
        f"{[path.name for path in seed_dirs]}"
    )

for seed_dir in seed_dirs:
    file_count = sum(1 for path in seed_dir.rglob("*") if path.is_file())
    print(f"{seed_dir.name}: {file_count:,} files")

## Compare vocabularies selected by frequency w.r.t. Wolpaw, In-Vocab MI (Speier), and OVMI

In [ ]:
ensure_package("matplotlib")

from collections import defaultdict
from pathlib import Path
import sys
from typing import Optional

from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
})


SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ovmi import load_subtlex_uk


SUBTLEX_CACHE_DIR = PROJECT_ROOT / "experiments" / "data" / "cache"
SUBTLEX_PATH = SUBTLEX_CACHE_DIR / "SUBTLEX-UK.xlsx"
SUBTLEX_REFERENCE = load_subtlex_uk(path=SUBTLEX_PATH)


def subtlex_frequencies_for_vocab(vocab_words: np.ndarray) -> np.ndarray:
    return np.array([SUBTLEX_REFERENCE.get(str(word), 0.0) for word in vocab_words], dtype=np.float64)


def load_prediction_run(run_dir: Path, split: str = "test") -> dict[str, np.ndarray]:
    """Load one seed/run from the downloaded Hugging Face snapshot."""
    with np.load(run_dir / f"{split}_predictions_best.npz", allow_pickle=True) as predictions:
        subject = predictions["subject"].astype(str)
        target_word = predictions["target_word"].astype(str)
        pred_embedding = predictions["pred_embedding"].astype(np.float64)
        sentence_idx_of_word = predictions["sentence_idx_of_word"].astype(np.int64)
    with np.load(run_dir / "vocab_embeddings.npz", allow_pickle=True) as vocab:
        vocab_words = vocab["word"].astype(str)
        vocab_embeddings = vocab["t5_embedding"].astype(np.float64)
    with np.load(run_dir / "vocab_metadata.npz", allow_pickle=True) as metadata:
        word_frequency = metadata["word_frequency"].astype(np.float64)

    return {
        "subject": subject,
        "target_word": target_word,
        "pred_embedding": pred_embedding,
        "sentence_idx_of_word": sentence_idx_of_word,
        "vocab_words": vocab_words,
        "vocab_embeddings": vocab_embeddings,
        "word_frequency": word_frequency,
    }


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)

In [ ]:
def wolpaw_bits(correct_probability: float, vocab_size: int) -> float:
    if vocab_size <= 1:
        return 0.0
    if correct_probability <= 0.0:
        return float(np.log2(vocab_size) + np.log2(1.0 / (vocab_size - 1)))
    if correct_probability >= 1.0:
        return float(np.log2(vocab_size))
    return float(
        np.log2(vocab_size)
        + correct_probability * np.log2(correct_probability)
        + (1.0 - correct_probability)
        * np.log2((1.0 - correct_probability) / (vocab_size - 1))
    )


def in_vocab_mi(correct_probability: float, word_probabilities: np.ndarray) -> float:
    """Speier-style MI inside the selected vocabulary."""
    vocab_size = len(word_probabilities)
    if vocab_size <= 1 or correct_probability <= 0.0:
        return 0.0
    if correct_probability >= 1.0:
        positive = word_probabilities > 0
        return float(-np.sum(word_probabilities[positive] * np.log2(word_probabilities[positive])))

    output_probabilities = (
        correct_probability * word_probabilities
        + (1.0 - correct_probability) * (1.0 - word_probabilities) / (vocab_size - 1)
    )
    positive = output_probabilities > 0
    output_entropy = -np.sum(output_probabilities[positive] * np.log2(output_probabilities[positive]))
    conditional_entropy = -(
        correct_probability * np.log2(correct_probability)
        + (1.0 - correct_probability) * np.log2((1.0 - correct_probability) / (vocab_size - 1))
    )
    return float(max(0.0, output_entropy - conditional_entropy))


def subset_accuracies(
    selected: np.ndarray,
    similarity: np.ndarray,
    rows_by_word: dict[int, np.ndarray],
    instance_labels: np.ndarray,
) -> np.ndarray:
    """Decode selected words against selected retrieval targets."""
    rows = np.concatenate([rows_by_word[int(i)] for i in selected if int(i) in rows_by_word])
    local_to_selected = np.full(similarity.shape[1], -1, dtype=np.int64)
    local_to_selected[selected] = np.arange(len(selected))

    true = local_to_selected[instance_labels[rows]]
    predicted = similarity[np.ix_(rows, selected)].argmax(axis=1)

    correct = np.zeros(len(selected), dtype=np.float64)
    total = np.zeros(len(selected), dtype=np.float64)
    np.add.at(total, true, 1)
    np.add.at(correct, true, predicted == true)
    return np.divide(correct, total, out=np.zeros_like(correct), where=total > 0)

In [ ]:
def compute_three_metric_traces(
    run_dir: Path,
    max_vocab: Optional[int] = None,
    min_word_instances: int = 5,
) -> dict[str, np.ndarray]:
    data = load_prediction_run(run_dir)
    vocab_words = data["vocab_words"]
    word_to_vocab_index = {word: i for i, word in enumerate(vocab_words)}

    instances_by_word = defaultdict(list)
    for word, embedding in zip(data["target_word"], data["pred_embedding"]):
        instances_by_word[word].append(embedding)

    eligible_words = [
        word
        for word, embeddings in instances_by_word.items()
        if word in word_to_vocab_index and len(embeddings) >= min_word_instances
    ]
    if not eligible_words:
        raise ValueError(f"No words in {run_dir} have at least {min_word_instances} instances.")

    test_indices = np.array(
        sorted(word_to_vocab_index[word] for word in eligible_words),
        dtype=np.int64,
    )

    instance_chunks = []
    instance_labels = []
    rows_by_word = {}
    row_start = 0
    for local_index, vocab_index in enumerate(test_indices):
        word_instances = np.stack(instances_by_word[vocab_words[vocab_index]])
        row_stop = row_start + len(word_instances)
        rows_by_word[local_index] = np.arange(row_start, row_stop)
        instance_chunks.append(word_instances)
        instance_labels.extend([local_index] * len(word_instances))
        row_start = row_stop

    all_instances = np.concatenate(instance_chunks)
    instance_labels = np.array(instance_labels, dtype=np.int64)
    retrieval_embeddings = data["vocab_embeddings"][test_indices]

    with np.errstate(all="ignore"):
        similarity = normalize_rows(all_instances) @ normalize_rows(retrieval_embeddings).T

    frequencies = subtlex_frequencies_for_vocab(data["vocab_words"])
    total_frequency = frequencies.sum()
    order = np.argsort(frequencies[test_indices])[::-1]
    if max_vocab is not None:
        order = order[:max_vocab]

    selected = []
    traces = {"wolpaw": [], "speier": [], "ovmi": [], "coverage": []}
    for local_index in order:
        selected.append(int(local_index))
        selected_array = np.array(selected, dtype=np.int64)
        selected_frequencies = frequencies[test_indices[selected_array]]
        coverage = selected_frequencies.sum() / total_frequency

        if len(selected) <= 1 or selected_frequencies.sum() == 0:
            wolpaw = speier = ovmi_score = 0.0
        else:
            per_word_accuracy = subset_accuracies(selected_array, similarity, rows_by_word, instance_labels)
            correct_probability = float(per_word_accuracy.mean())
            word_probabilities = selected_frequencies / selected_frequencies.sum()
            wolpaw = wolpaw_bits(correct_probability, len(selected))
            speier = in_vocab_mi(correct_probability, word_probabilities)
            ovmi_score = coverage * speier

        traces["wolpaw"].append(wolpaw)
        traces["speier"].append(speier)
        traces["ovmi"].append(ovmi_score)
        traces["coverage"].append(coverage)

    return {name: np.array(values, dtype=np.float64) for name, values in traces.items()}

In [ ]:
MAX_VOCAB = 110
MIN_WORD_INSTANCES = 1

run_results = {}
peak_rows = []
for seed_dir in seed_dirs:
    traces = compute_three_metric_traces(
        seed_dir,
        max_vocab=MAX_VOCAB,
        min_word_instances=MIN_WORD_INSTANCES,
    )
    run_results[seed_dir.name] = traces

    for metric in ["wolpaw", "speier", "ovmi"]:
        peak_index = int(np.nanargmax(traces[metric]))
        peak_rows.append({
            "seed": seed_dir.name,
            "metric": metric,
            "peak_bits": traces[metric][peak_index],
            "peak_vocab_size": peak_index + 1,
        })

peak_table = pd.DataFrame(peak_rows)
peak_table

In [ ]:
def stack_metric(metric: str) -> np.ndarray:
    traces = [result[metric] for result in run_results.values()]
    max_length = max(len(trace) for trace in traces)
    stacked = np.full((len(traces), max_length), np.nan)
    for row, trace in enumerate(traces):
        stacked[row, : len(trace)] = trace
    return stacked


summary = {}
for metric in ["wolpaw", "speier", "ovmi", "coverage"]:
    stacked = stack_metric(metric)
    n = np.sum(~np.isnan(stacked), axis=0)
    summary[metric] = {
        "mean": np.nanmean(stacked, axis=0),
        "se": np.nanstd(stacked, axis=0, ddof=1) / np.sqrt(n),
    }

mean_peak_rows = []
for metric in ["wolpaw", "speier", "ovmi"]:
    peak_index = int(np.nanargmax(summary[metric]["mean"]))
    mean_peak_rows.append({
        "metric": metric,
        "mean_peak_bits": summary[metric]["mean"][peak_index],
        "peak_vocab_size": peak_index + 1,
    })

pd.DataFrame(mean_peak_rows)

In [ ]:
colors = {
    "wolpaw": "#2ca02c",
    "speier": "#ff7f0e",
    "ovmi": "#1f77b4",
    "coverage": "#9467bd",
}
labels = {
    "wolpaw": "Wolpaw",
    "speier": "In-Vocab MI (Speier)",
    "ovmi": "OVMI",
    "coverage": "Coverage C(S)",
}


def plot_mean_with_se(axis, sizes, metric, linewidth=1.5, alpha=0.18):
    mean = summary[metric]["mean"]
    se = summary[metric]["se"]
    axis.plot(sizes, mean, "-", color=colors[metric], linewidth=linewidth, label=labels[metric])
    axis.fill_between(sizes, mean - se, mean + se, color=colors[metric], alpha=alpha, linewidth=0)


def style_arrow_axes(axis):
    """Match the arrowed-axis styling used in the original plotting script."""
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    arrowprops = {
        "arrowstyle": "-|>",
        "color": "black",
        "linewidth": 0.8,
        "mutation_scale": 8,
        "shrinkA": 0,
        "shrinkB": 0,
    }
    axis.annotate(
        "",
        xy=(1.02, 0),
        xytext=(0.97, 0),
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops=arrowprops,
        clip_on=False,
    )
    axis.annotate(
        "",
        xy=(0, 1.04),
        xytext=(0, 0.96),
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops=arrowprops,
        clip_on=False,
    )


sizes = np.arange(1, len(summary["ovmi"]["mean"]) + 1)
fig, (ax_metrics, ax_decomposition) = plt.subplots(1, 2, figsize=(8, 3))

for metric in ["wolpaw", "speier", "ovmi"]:
    plot_mean_with_se(ax_metrics, sizes, metric)
    peak_index = int(np.nanargmax(summary[metric]["mean"]))
    ax_metrics.plot(
        sizes[peak_index],
        summary[metric]["mean"][peak_index],
        "*",
        color=colors[metric],
        markeredgecolor="black",
        markersize=10,
        zorder=5,
    )

ax_metrics.set_xlabel(r"Vocabulary size $V$ (selected by frequency)")
ax_metrics.set_ylabel("bits / word attempt")
ax_metrics.set_ylim(bottom=0)
ax_metrics.grid(axis="y", alpha=0.3)
ax_metrics.grid(axis="x", alpha=0.3)
style_arrow_axes(ax_metrics)

for metric in ["speier", "ovmi"]:
    plot_mean_with_se(ax_decomposition, sizes, metric)
ax_decomposition.set_xlabel(r"Vocabulary size $V$ (selected by frequency)")
ax_decomposition.set_ylabel("bits / word attempt")
ax_decomposition.set_ylim(bottom=0)
ax_decomposition.grid(axis="y", alpha=0.3)
ax_decomposition.grid(axis="x", alpha=0.3)
style_arrow_axes(ax_decomposition)

ax_coverage = ax_decomposition.twinx()
plot_mean_with_se(ax_coverage, sizes, "coverage")
ax_coverage.set_ylabel("Coverage", color=colors["coverage"])
ax_coverage.tick_params(axis="y", labelcolor=colors["coverage"])
ax_coverage.set_ylim(0, 1)
ax_coverage.spines["top"].set_visible(False)
ax_coverage.spines["right"].set_visible(False)

legend_handles = [
    Line2D([], [], color=colors["wolpaw"], linewidth=1.5),
    Line2D([], [], color=colors["speier"], linewidth=1.5),
    Line2D([], [], color=colors["ovmi"], linewidth=1.5),
    Line2D([], [], color=colors["coverage"], linewidth=1.5),
]
legend_labels = [labels["wolpaw"], labels["speier"], labels["ovmi"], labels["coverage"]]
fig.legend(legend_handles, legend_labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.0), frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## True Information Transfer is Relative to a Reference Distribution

In [ ]:
ensure_package("wordcloud")

from matplotlib import font_manager
from wordcloud import WordCloud
import matplotlib

matplotlib.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "axes.linewidth": 1.2,
    "hatch.linewidth": 1.1,
})

BAR_MAX_VOCAB = 500
BAR_MIN_WORD_INSTANCES = 5
BAR_WORDCLOUD_TOP_N = 200
BAR_RECOMPUTE = False
BAR_CACHE_PATH = PROJECT_ROOT / "experiments" / "data" / "ovmi_bars_multiple_notebook.npz"
BAR_OUTPUT_PATH = PROJECT_ROOT / "experiments" / "figures" / "ovmi_bars_multiple"

MOSES_WORDS = [
    "Am", "Are", "Bad", "Bring", "Clean", "Closer", "Comfortable", "Coming",
    "Computer", "Do", "Faith", "Family", "Feel", "Glasses", "Going", "Good",
    "Goodbye", "Have", "Hello", "Help", "Here", "Hope", "How", "Hungry", "I",
    "Is", "It", "Like", "Music", "My", "Need", "No", "Not", "Nurse", "Okay",
    "Outside", "Please", "Right", "Success", "Tell", "That", "They", "Thirsty",
    "Tired", "Up", "Very", "What", "Where", "Yes", "You",
]
MOSES_PC = 0.471
WILLETT_PC = 0.951
MOSES_V = 50
UNIVERSAL_CORE_WORDS = [
    "all", "in", "some",
    "can", "it", "stop",
    "different", "like", "that",
    "do", "look", "turn",
    "finished", "make", "up",
    "get", "more", "want",
    "go", "not", "what",
    "good", "on", "when",
    "he", "open", "where",
    "help", "put", "who",
    "here", "same", "why",
    "I", "she", "you",
]
FUNCTION_WORDS = {
    "a", "an", "the",
    "and", "or", "but", "if", "because", "as", "while", "although",
    "for", "nor", "so", "yet",
    "about", "above", "across", "after", "against", "along", "among",
    "around", "at", "before", "behind", "below", "beneath", "beside",
    "between", "by", "down", "during", "from", "in", "inside", "into",
    "near", "of", "off", "on", "onto", "out", "outside", "over", "through",
    "to", "toward", "under", "until", "up", "upon", "with", "within",
    "without",
    "i", "me", "my", "mine", "myself", "we", "us", "our", "ours",
    "ourselves", "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs",
    "themselves",
    "this", "that", "these", "those", "who", "whom", "whose", "which",
    "what", "where", "when", "why", "how",
    "am", "are", "is", "was", "were", "be", "been", "being",
    "do", "does", "did", "doing", "have", "has", "had", "having",
    "can", "could", "may", "might", "must", "shall", "should", "will",
    "would",
    "not", "no", "yes", "all", "any", "both", "each", "either", "few",
    "many", "more", "most", "much", "neither", "some", "such",
    "s", "re", "ve", "ll", "d", "m", "t",
}
GREEDY_BAR_COLOR = "#8E44AD"
MOSES_BAR_COLOR = "#A6A6A6"
WILLETT_BAR_COLOR = "#B85C00"
MOSES_BAR_HATCH = "///"
WILLETT_BAR_HATCH = "\\\\"
MOSES_TITLE_COLOR = "#5F5F5F"


def bar_group_instances(words: np.ndarray, embeddings: np.ndarray) -> dict[str, np.ndarray]:
    grouped = defaultdict(list)
    for word, embedding in zip(words, embeddings):
        grouped[str(word)].append(embedding)
    return {word: np.stack(values) for word, values in grouped.items()}


def bar_find_shared_words(val_instances_list, test_instances_list, min_examples: int) -> set[str]:
    per_run = []
    for val_instances, test_instances in zip(val_instances_list, test_instances_list):
        per_run.append({
            word
            for word in val_instances
            if word in test_instances
            and len(val_instances[word]) >= min_examples
            and len(test_instances[word]) >= min_examples
        })
    return set.intersection(*per_run) if per_run else set()


def bar_prepare_split(data: dict[str, np.ndarray], test_indices: np.ndarray) -> dict[str, object]:
    instances_by_word = bar_group_instances(data["target_word"], data["pred_embedding"])
    vocab_words = data["vocab_words"]

    chunks = []
    labels = []
    rows_by_word = {}
    row_start = 0
    for local_index, vocab_index in enumerate(test_indices):
        word_instances = instances_by_word[str(vocab_words[vocab_index])]
        row_stop = row_start + len(word_instances)
        rows_by_word[local_index] = np.arange(row_start, row_stop)
        chunks.append(word_instances)
        labels.extend([local_index] * len(word_instances))
        row_start = row_stop

    all_instances = np.concatenate(chunks).astype(np.float64)
    retrieval = data["vocab_embeddings"][test_indices].astype(np.float64)
    with np.errstate(all="ignore"):
        similarity = normalize_rows(all_instances) @ normalize_rows(retrieval).T

    return {
        "similarity": similarity,
        "rows_by_word": rows_by_word,
        "labels": np.array(labels, dtype=np.int64),
        "instances_by_word": instances_by_word,
    }


def bar_pool_val_data(val_splits: list[dict[str, object]]) -> dict[str, object]:
    pooled_similarity = np.concatenate([split["similarity"] for split in val_splits], axis=0)
    pooled_labels = np.concatenate([split["labels"] for split in val_splits], axis=0)
    pooled_rows = {}
    offset = 0
    for split in val_splits:
        for local_index, rows in split["rows_by_word"].items():
            pooled_rows.setdefault(local_index, []).append(rows + offset)
        offset += split["similarity"].shape[0]
    pooled_rows = {local_index: np.concatenate(parts) for local_index, parts in pooled_rows.items()}
    return {"similarity": pooled_similarity, "rows_by_word": pooled_rows, "labels": pooled_labels}


def bar_pool_test_counts(test_instances_list) -> dict[str, int]:
    counts = {}
    for instances in test_instances_list:
        for word, embeddings in instances.items():
            counts[word] = counts.get(word, 0) + len(embeddings)
    return counts


def bar_pseudo_instances(counts: dict[str, int]) -> dict[str, list[None]]:
    return {word: [None] * count for word, count in counts.items()}


def _subtlex_lower_lookup() -> dict[str, float]:
    return {str(word).lower(): float(frequency) for word, frequency in SUBTLEX_REFERENCE.items()}


def build_moses_freq(vocab_words: np.ndarray, subtlex_freq: np.ndarray) -> np.ndarray:
    moses_lower = {word.lower() for word in MOSES_WORDS}
    freq = np.zeros(len(vocab_words), dtype=np.float64)
    for i, word in enumerate(vocab_words):
        if str(word).lower() in moses_lower:
            freq[i] = subtlex_freq[i]
    print(f"Moses words with non-zero SUBTLEX in model vocab: {int((freq > 0).sum())}/{MOSES_V}")
    return freq


def build_sherlock_freq(vocab_words: np.ndarray, test_instances: dict[str, object]) -> np.ndarray:
    word_to_index = {str(word): i for i, word in enumerate(vocab_words)}
    freq = np.zeros(len(vocab_words), dtype=np.float64)
    for word, embeddings in test_instances.items():
        index = word_to_index.get(str(word))
        if index is not None:
            freq[index] = len(embeddings)
    print(f"Sherlock reference: {int((freq > 0).sum())} vocab words with non-zero test count, total={int(freq.sum())} instances")
    return freq


def build_universal_core_freq(vocab_words: np.ndarray, subtlex_freq: np.ndarray) -> np.ndarray:
    ucv_lower = {word.lower() for word in UNIVERSAL_CORE_WORDS}
    freq = np.zeros(len(vocab_words), dtype=np.float64)
    for i, word in enumerate(vocab_words):
        if str(word).lower() in ucv_lower:
            freq[i] = subtlex_freq[i]
    print(f"Universal core words with non-zero SUBTLEX in model vocab: {int((freq > 0).sum())}/{len(UNIVERSAL_CORE_WORDS)}")
    return freq


def _symmetric_scalar_ovmi(freqs: np.ndarray, coverage_denominator: float, vocab_size: int, pc: float) -> float:
    freq_total = float(freqs.sum())
    if coverage_denominator <= 0 or freq_total <= 0 or vocab_size <= 1:
        return 0.0
    coverage = freq_total / coverage_denominator
    p_x = freqs / freq_total
    pc = float(np.clip(pc, 1e-15, 1.0 - 1e-15))
    q_y = (1.0 - pc) / (vocab_size - 1) + p_x * (pc - (1.0 - pc) / (vocab_size - 1))
    h_y = -float(np.sum(q_y[q_y > 0] * np.log2(q_y[q_y > 0])))
    h_yx = -pc * np.log2(pc) - (1.0 - pc) * np.log2((1.0 - pc) / (vocab_size - 1))
    return coverage * max(0.0, h_y - h_yx)


def _moses_subtlex_freqs() -> np.ndarray:
    lookup = _subtlex_lower_lookup()
    return np.array([lookup.get(word.lower(), 0.0) for word in MOSES_WORDS], dtype=np.float64)


def compute_moses_ovmi(subtlex_freq_total: float) -> float:
    return _symmetric_scalar_ovmi(_moses_subtlex_freqs(), subtlex_freq_total, MOSES_V, MOSES_PC)


def compute_willett_ovmi(subtlex_freq_total: float) -> float:
    return _symmetric_scalar_ovmi(_moses_subtlex_freqs(), subtlex_freq_total, MOSES_V, WILLETT_PC)


def moses_invasive_ovmi_under_moses_ref() -> float:
    freqs = _moses_subtlex_freqs()
    return _symmetric_scalar_ovmi(freqs, freqs.sum(), MOSES_V, MOSES_PC)


def willett_ovmi_under_moses_ref() -> float:
    freqs = _moses_subtlex_freqs()
    return _symmetric_scalar_ovmi(freqs, freqs.sum(), MOSES_V, WILLETT_PC)


def _moses_word_counts(test_instances: dict[str, object]) -> tuple[np.ndarray, float]:
    counts = {}
    for word, embeddings in test_instances.items():
        key = str(word).lower()
        counts[key] = counts.get(key, 0) + len(embeddings)
    return np.array([counts.get(word.lower(), 0) for word in MOSES_WORDS], dtype=np.float64), sum(counts.values())


def moses_invasive_ovmi_under_sherlock_ref(test_instances: dict[str, object]) -> float:
    freqs, total = _moses_word_counts(test_instances)
    return _symmetric_scalar_ovmi(freqs, total, MOSES_V, MOSES_PC)


def willett_ovmi_under_sherlock_ref(test_instances: dict[str, object]) -> float:
    freqs, total = _moses_word_counts(test_instances)
    return _symmetric_scalar_ovmi(freqs, total, MOSES_V, WILLETT_PC)


def _moses_freqs_under_universal_core_ref() -> tuple[np.ndarray, float]:
    lookup = _subtlex_lower_lookup()
    ucv_lower = {word.lower() for word in UNIVERSAL_CORE_WORDS}
    reference_total = sum(lookup.get(word, 0.0) for word in ucv_lower)
    freqs = np.array([
        lookup.get(word.lower(), 0.0) if word.lower() in ucv_lower else 0.0
        for word in MOSES_WORDS
    ], dtype=np.float64)
    return freqs, reference_total


def moses_invasive_ovmi_under_universal_core_ref() -> float:
    freqs, reference_total = _moses_freqs_under_universal_core_ref()
    return _symmetric_scalar_ovmi(freqs, reference_total, MOSES_V, MOSES_PC)


def willett_ovmi_under_universal_core_ref() -> float:
    freqs, reference_total = _moses_freqs_under_universal_core_ref()
    return _symmetric_scalar_ovmi(freqs, reference_total, MOSES_V, WILLETT_PC)


def bar_subset_accuracies(selected: np.ndarray, similarity: np.ndarray, rows_by_word: dict[int, np.ndarray], labels: np.ndarray) -> np.ndarray:
    parts = [rows_by_word[int(local_index)] for local_index in selected if int(local_index) in rows_by_word]
    if not parts:
        return np.zeros(len(selected), dtype=np.float64)
    rows = np.concatenate(parts)
    local_to_selected = np.full(similarity.shape[1], -1, dtype=np.int64)
    local_to_selected[selected] = np.arange(len(selected))
    true = local_to_selected[labels[rows]]
    predicted = similarity[np.ix_(rows, selected)].argmax(axis=1)

    correct = np.zeros(len(selected), dtype=np.float64)
    total = np.zeros(len(selected), dtype=np.float64)
    np.add.at(total, true, 1)
    np.add.at(correct, true, predicted == true)
    return np.divide(correct, total, out=np.zeros_like(correct), where=total > 0)


def bar_ovmi_for_subset(selected, split: dict[str, object], reference_frequency: np.ndarray, test_indices: np.ndarray) -> float:
    selected = np.array(selected, dtype=np.int64)
    if len(selected) <= 1:
        return 0.0
    selected_frequency = reference_frequency[test_indices[selected]].astype(np.float64)
    if selected_frequency.sum() <= 0 or reference_frequency.sum() <= 0:
        return 0.0
    accuracies = bar_subset_accuracies(selected, split["similarity"], split["rows_by_word"], split["labels"])
    coverage = selected_frequency.sum() / reference_frequency.sum()
    word_probabilities = selected_frequency / selected_frequency.sum()
    return coverage * in_vocab_mi(float(accuracies.mean()), word_probabilities)


def bar_trace_from_order(order, split: dict[str, object], reference_frequency: np.ndarray, test_indices: np.ndarray) -> np.ndarray:
    selected = []
    trace = []
    for local_index in order:
        selected.append(int(local_index))
        trace.append(bar_ovmi_for_subset(selected, split, reference_frequency, test_indices))
    return np.array(trace, dtype=np.float64)


def bar_greedy_order_on_validation(val_split: dict[str, object], reference_frequency: np.ndarray, test_indices: np.ndarray, max_vocab: int) -> tuple[list[int], np.ndarray]:
    eligible = [
        local_index
        for local_index, vocab_index in enumerate(test_indices)
        if reference_frequency[vocab_index] > 0 and local_index in val_split["rows_by_word"]
    ]
    n_steps = min(len(eligible), max_vocab)
    if n_steps == 0:
        return [], np.array([], dtype=np.float64)

    selected = []
    selected_set = set()
    trace = []
    for step in range(n_steps):
        remaining = [local_index for local_index in eligible if local_index not in selected_set]
        scores = [
            bar_ovmi_for_subset(selected + [candidate], val_split, reference_frequency, test_indices)
            for candidate in remaining
        ]
        best_position = int(np.argmax(scores))
        best_candidate = remaining[best_position]
        selected.append(best_candidate)
        selected_set.add(best_candidate)
        trace.append(float(scores[best_position]))
        vocab_size = step + 1
        if vocab_size <= 20 or vocab_size % 10 == 0:
            print(f"  step {vocab_size:4d}: +word[{test_indices[best_candidate]}]  OV-MI={trace[-1]:.4f}")
    return selected, np.array(trace, dtype=np.float64)


def topk_order_by_subtlex(subtlex_freq: np.ndarray, test_indices: np.ndarray, eligible_local: list[int], max_vocab: int) -> list[int]:
    eligible = np.asarray(eligible_local, dtype=np.int64)
    if len(eligible) == 0:
        return []
    scores = subtlex_freq[test_indices].astype(np.float64)
    order = eligible[np.argsort(scores[eligible])[::-1]].tolist()
    return order[:max_vocab]


def _mean_se(values: np.ndarray) -> tuple[float, float]:
    mean = float(np.mean(values))
    se = float(np.std(values, ddof=1) / np.sqrt(len(values))) if len(values) > 1 else 0.0
    return mean, se


def evaluate_reference_multirun(label: str, reference_frequency: np.ndarray, subtlex_frequency: np.ndarray, pooled_val_split: dict[str, object], test_splits: list[dict[str, object]], test_indices: np.ndarray, max_vocab: int) -> dict[str, object]:
    print(f"\n--- {label} reference ---")
    print("  Greedy on pooled val...")
    val_greedy_order, val_greedy_trace = bar_greedy_order_on_validation(
        pooled_val_split, reference_frequency, test_indices, max_vocab=max_vocab
    )
    if len(val_greedy_trace) == 0:
        raise ValueError(f"No eligible words under {label} reference")
    greedy_peak_V = int(np.argmax(val_greedy_trace)) + 1
    greedy_order_trunc = val_greedy_order[:greedy_peak_V]

    eligible_local = [
        local_index
        for local_index, vocab_index in enumerate(test_indices)
        if reference_frequency[vocab_index] > 0
    ]
    topk_order_full = topk_order_by_subtlex(subtlex_frequency, test_indices, eligible_local, max_vocab)
    val_topk_trace = bar_trace_from_order(topk_order_full, pooled_val_split, reference_frequency, test_indices)
    if len(val_topk_trace) == 0:
        raise ValueError(f"No top-k candidates under {label} reference")
    topk_peak_V = int(np.argmax(val_topk_trace)) + 1
    topk_order_trunc = topk_order_full[:topk_peak_V]

    test_greedy_values = []
    test_topk_values = []
    for split in test_splits:
        test_greedy_values.append(float(bar_trace_from_order(greedy_order_trunc, split, reference_frequency, test_indices)[-1]))
        test_topk_values.append(float(bar_trace_from_order(topk_order_trunc, split, reference_frequency, test_indices)[-1]))

    test_greedy_arr = np.array(test_greedy_values, dtype=np.float64)
    test_topk_arr = np.array(test_topk_values, dtype=np.float64)
    greedy_mean, greedy_se = _mean_se(test_greedy_arr)
    topk_mean, topk_se = _mean_se(test_topk_arr)
    print(f"  Greedy: peak_V={greedy_peak_V}, test OV-MI = {greedy_mean:.4f} +/- {greedy_se:.4f} (N={len(test_greedy_arr)})")
    print(f"  Top-k:  peak_V={topk_peak_V}, test OV-MI = {topk_mean:.4f} +/- {topk_se:.4f} (N={len(test_topk_arr)})")

    return {
        "val_greedy_trace": val_greedy_trace,
        "val_topk_trace": val_topk_trace,
        "greedy_peak_V": greedy_peak_V,
        "topk_peak_V": topk_peak_V,
        "test_greedy_per_run": test_greedy_arr,
        "test_topk_per_run": test_topk_arr,
        "test_greedy_mean": greedy_mean,
        "test_greedy_se": greedy_se,
        "test_topk_mean": topk_mean,
        "test_topk_se": topk_se,
        "n_eligible": len(eligible_local),
    }


def build_wordcloud_freq_map(vocab_words: np.ndarray, freq_arr: np.ndarray, top_n: int = 200, exclude_function_words: bool = True) -> dict[str, float]:
    items = []
    for word, frequency in zip(vocab_words, freq_arr):
        word = str(word).strip()
        frequency = float(frequency)
        if exclude_function_words and word.lower() in FUNCTION_WORDS:
            continue
        if word and frequency > 0:
            items.append((word, frequency))
    items.sort(key=lambda item: item[1], reverse=True)
    return dict(items[:top_n])


def draw_wordcloud(axis, freq_map: dict[str, float]) -> None:
    axis.set_axis_off()
    if not freq_map:
        axis.text(0.5, 0.5, "No words", ha="center", va="center", fontsize=9)
        return

    def gray_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
        shade = random_state.randint(26, 52) if random_state else 38
        return f"hsl(215, 18%, {shade}%)"

    font_family = matplotlib.rcParams.get("font.family", ["sans-serif"])
    font_path = font_manager.findfont(
        font_manager.FontProperties(family=font_family),
        fallback_to_default=True,
    )
    wordcloud = WordCloud(
        width=1100,
        height=350,
        margin=0,
        relative_scaling=0.35,
        font_step=1,
        background_color="white",
        prefer_horizontal=0.9,
        collocations=False,
        font_path=font_path,
        color_func=gray_color_func,
    )
    axis.imshow(wordcloud.generate_from_frequencies(freq_map), interpolation="bilinear")
    axis.set_axis_off()


def style_y_axis_arrow(axis) -> None:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_linewidth(1.2)
    axis.spines["bottom"].set_linewidth(1.2)
    axis.annotate(
        "",
        xy=(0, 1.04),
        xytext=(0, 0.96),
        xycoords="axes fraction",
        textcoords="axes fraction",
        arrowprops={
            "arrowstyle": "-|>",
            "color": "black",
            "linewidth": 1.2,
            "mutation_scale": 10,
            "shrinkA": 0,
            "shrinkB": 0,
        },
        clip_on=False,
    )


def plot_bars_multirun(subtlex_ref, moses_ref, sherlock_ref, universal_core_ref, wordcloud_freqs, output_path: Path):
    fig = plt.figure(figsize=(9, 3), constrained_layout=True)
    fig.set_constrained_layout_pads(h_pad=0.02, w_pad=0.02, hspace=0.02, wspace=0.02)
    grid = fig.add_gridspec(
        3,
        5,
        height_ratios=[0.18, 2.7, 1.45],
        width_ratios=[1, 1, 0.06, 1, 1],
    )

    header_independent = fig.add_subplot(grid[0, 0:2])
    header_aligned = fig.add_subplot(grid[0, 3:5])
    for axis, label in [
        (header_independent, "Independent reference distributions"),
        (header_aligned, "Design-aligned reference distributions"),
    ]:
        axis.axis("off")
        axis.text(0.5, 0.18, label, ha="center", va="center", fontsize=10, fontweight="normal", color="black")

    divider_axis = fig.add_subplot(grid[:, 2])
    divider_axis.axis("off")
    divider_axis.plot([0.5, 0.5], [0.0, 1.0], color="0.6", linewidth=0.8, linestyle="--", transform=divider_axis.transAxes, clip_on=False)

    plot_columns = [0, 1, 3, 4]
    bar_axes = [fig.add_subplot(grid[1, column]) for column in plot_columns]
    wordcloud_axes = [fig.add_subplot(grid[2, column]) for column in plot_columns]
    methods = ["MEG-XL$^\\mathbf{*}$", "Moses", "Willett"]
    bar_fill_colors = [GREEDY_BAR_COLOR, MOSES_BAR_COLOR, WILLETT_BAR_COLOR]
    bar_hatches = [None, MOSES_BAR_HATCH, WILLETT_BAR_HATCH]
    panels = [
        ("subtlex", bar_axes[0], subtlex_ref, r"$p = \mathrm{SUBTLEX}$", True, "black"),
        ("universal_core", bar_axes[1], universal_core_ref, r"$p = \mathrm{UCV}$", False, "black"),
        ("moses", bar_axes[2], moses_ref, r"$p = \mathrm{Moses}$", False, MOSES_TITLE_COLOR),
        ("sherlock", bar_axes[3], sherlock_ref, r"$p = \mathrm{Sherlock}$", False, GREEDY_BAR_COLOR),
    ]

    for key, axis, panel, title, show_ylabel, title_color in panels:
        values = [panel["test_greedy_mean"], panel["moses_ovmi"], panel["willett_ovmi"]]
        errors = [panel["test_greedy_se"], 0.0, 0.0]
        x = np.arange(len(methods))
        bars = axis.bar(
            x,
            values,
            yerr=errors,
            color=bar_fill_colors,
            width=0.62,
            edgecolor="black",
            linewidth=0.8,
            zorder=3,
            error_kw={"elinewidth": 1.5, "ecolor": "black", "capsize": 4.0, "capthick": 1.5},
        )
        for bar, hatch in zip(bars, bar_hatches):
            if hatch is not None:
                bar.set_hatch(hatch)
        axis.set_xticks(x)
        axis.set_xticklabels(methods)
        for tick in axis.get_xticklabels():
            tick.set_color("black")
            tick.set_fontweight("bold")
        axis.set_title(title, color=title_color)
        axis.set_ylabel("bits / word attempt" if show_ylabel else "")
        top = max(value + error for value, error in zip(values, errors))
        axis.set_ylim(0, (top * 1.38) if top > 0 else 1.0)
        axis.set_axisbelow(True)
        axis.grid(axis="y", alpha=0.22, zorder=0)
        axis.axhline(0, color="black", lw=1.0)
        axis.tick_params(axis="both", colors="black", width=1.2)
        # axis.tick_params(axis="x", rotation=15)
        style_y_axis_arrow(axis)

        tags = [
            (f"{panel['test_greedy_mean']:.3f}", f"V={panel['greedy_peak_V']}"),
            (f"{panel['moses_ovmi']:.3f}", ""),
            (f"{panel['willett_ovmi']:.3f}", ""),
        ]
        for xi, value, error, (main, extra) in zip(x, values, errors, tags):
            axis.annotate(
                main + (f"\n{extra}" if extra else ""),
                (xi, value + error),
                textcoords="offset points",
                xytext=(0, 3),
                ha="center",
                fontsize=7.2,
                fontweight="bold",
                color="black",
                linespacing=0.9,
            )

    draw_wordcloud(wordcloud_axes[0], wordcloud_freqs.get("subtlex", {}))
    draw_wordcloud(wordcloud_axes[1], wordcloud_freqs.get("universal_core", {}))
    draw_wordcloud(wordcloud_axes[2], wordcloud_freqs.get("moses", {}))
    draw_wordcloud(wordcloud_axes[3], wordcloud_freqs.get("sherlock", {}))

    fig.text(0.012, 0.13, r"Ref. $p(x)$", rotation=90, ha="center", va="center", fontsize=11)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    for extension in [".pdf", ".png"]:
        fig.savefig(output_path.with_suffix(extension), dpi=300, bbox_inches="tight")
        print(f"Saved: {output_path.with_suffix(extension)}")
    return fig


PANEL_SCALAR_KEYS = (
    "moses_ovmi",
    "willett_ovmi",
    "test_greedy_mean",
    "test_greedy_se",
    "test_topk_mean",
    "test_topk_se",
    "greedy_peak_V",
    "topk_peak_V",
)
PANEL_ARRAY_KEYS = ("test_greedy_per_run", "test_topk_per_run", "val_greedy_trace", "val_topk_trace")


def _panel_from_cache(cache, prefix: str) -> dict[str, object]:
    panel = {
        key: float(cache[f"{prefix}_{key}"])
        for key in PANEL_SCALAR_KEYS
        if not key.endswith("_V") and f"{prefix}_{key}" in cache.files
    }
    panel.update({key: int(cache[f"{prefix}_{key}"]) for key in ("greedy_peak_V", "topk_peak_V")})
    for key in PANEL_ARRAY_KEYS:
        panel[key] = np.asarray(cache[f"{prefix}_{key}"])
    return panel


def _save_panel_cache(path: Path, run_ids: list[str], panels: dict[str, dict[str, object]], vocab_words: np.ndarray, frequencies: dict[str, np.ndarray]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "run_ids": np.array(run_ids, dtype=str),
        "bar_max_vocab": np.array(BAR_MAX_VOCAB),
        "bar_min_word_instances": np.array(BAR_MIN_WORD_INSTANCES),
        "vocab_words": np.asarray(vocab_words),
    }
    for key, value in frequencies.items():
        payload[f"{key}_freq"] = np.asarray(value)
    for prefix, panel in panels.items():
        for key in PANEL_SCALAR_KEYS + PANEL_ARRAY_KEYS:
            payload[f"{prefix}_{key}"] = np.asarray(panel[key])
    np.savez(path, **payload)


bar_run_ids = [path.name for path in seed_dirs]
bar_cache = None
bar_required = {
    "run_ids", "bar_max_vocab", "bar_min_word_instances", "vocab_words",
    "subtlex_freq", "moses_freq", "sherlock_freq", "universal_core_freq",
}
for prefix in ["subtlex_ref", "moses_ref", "sherlock_ref", "universal_core_ref"]:
    for key in PANEL_SCALAR_KEYS + PANEL_ARRAY_KEYS:
        bar_required.add(f"{prefix}_{key}")

use_bar_cache = BAR_CACHE_PATH.exists() and not BAR_RECOMPUTE
if use_bar_cache:
    bar_cache = np.load(BAR_CACHE_PATH, allow_pickle=True)
    cache_files = set(bar_cache.files)
    willett_keys = {f"{prefix}_willett_ovmi" for prefix in ["subtlex_ref", "moses_ref", "sherlock_ref", "universal_core_ref"]}
    required_without_willett = bar_required - willett_keys
    use_bar_cache = (
        required_without_willett.issubset(cache_files)
        and list(bar_cache["run_ids"].astype(str)) == bar_run_ids
        and int(bar_cache["bar_max_vocab"]) == BAR_MAX_VOCAB
        and int(bar_cache["bar_min_word_instances"]) == BAR_MIN_WORD_INSTANCES
    )
    if use_bar_cache:
        print(f"Loading cached bar-plot results from {BAR_CACHE_PATH}")
    else:
        print("Existing bar-plot cache is stale or incomplete; recomputing.")

if use_bar_cache:
    subtlex_ref = _panel_from_cache(bar_cache, "subtlex_ref")
    moses_ref = _panel_from_cache(bar_cache, "moses_ref")
    sherlock_ref = _panel_from_cache(bar_cache, "sherlock_ref")
    universal_core_ref = _panel_from_cache(bar_cache, "universal_core_ref")
    bar_vocab_words = np.asarray(bar_cache["vocab_words"]).astype(str)
    bar_subtlex_freq = np.asarray(bar_cache["subtlex_freq"])
    bar_moses_freq = np.asarray(bar_cache["moses_freq"])
    bar_sherlock_freq = np.asarray(bar_cache["sherlock_freq"])
    bar_universal_core_freq = np.asarray(bar_cache["universal_core_freq"])
    if "willett_ovmi" not in subtlex_ref:
        subtlex_ref["willett_ovmi"] = compute_willett_ovmi(bar_subtlex_freq.sum())
        moses_ref["willett_ovmi"] = willett_ovmi_under_moses_ref()
        pooled_pseudo_test = bar_pseudo_instances({
            str(word): int(count)
            for word, count in zip(bar_vocab_words, bar_sherlock_freq)
            if count > 0
        })
        sherlock_ref["willett_ovmi"] = willett_ovmi_under_sherlock_ref(pooled_pseudo_test)
        universal_core_ref["willett_ovmi"] = willett_ovmi_under_universal_core_ref()
else:
    bar_val_instances_list = []
    bar_test_instances_list = []
    bar_val_data_list = []
    bar_test_data_list = []
    bar_vocab_words = None

    for seed_dir in seed_dirs:
        print(f"\nLoading run {seed_dir.name}...")
        val_data = load_prediction_run(seed_dir, split="val")
        test_data = load_prediction_run(seed_dir, split="test")
        if bar_vocab_words is None:
            bar_vocab_words = val_data["vocab_words"]
        elif len(val_data["vocab_words"]) != len(bar_vocab_words) or np.any(val_data["vocab_words"] != bar_vocab_words):
            raise ValueError(f"Run {seed_dir.name} does not share the same vocabulary ordering.")
        bar_val_data_list.append(val_data)
        bar_test_data_list.append(test_data)
        val_instances = bar_group_instances(val_data["target_word"], val_data["pred_embedding"])
        test_instances = bar_group_instances(test_data["target_word"], test_data["pred_embedding"])
        print(f"  Val:  {sum(len(values) for values in val_instances.values())} instances, {len(val_instances)} words")
        print(f"  Test: {sum(len(values) for values in test_instances.values())} instances, {len(test_instances)} words")
        bar_val_instances_list.append(val_instances)
        bar_test_instances_list.append(test_instances)

    bar_subtlex_freq = subtlex_frequencies_for_vocab(bar_vocab_words)
    shared_words = bar_find_shared_words(bar_val_instances_list, bar_test_instances_list, BAR_MIN_WORD_INSTANCES)
    word_to_index = {str(word): i for i, word in enumerate(bar_vocab_words)}
    bar_test_indices = np.array(sorted(word_to_index[word] for word in shared_words if word in word_to_index), dtype=np.int64)
    print(f"\nShared vocabulary (intersection across {len(seed_dirs)} runs, >={BAR_MIN_WORD_INSTANCES} instances in val AND test of every run): {len(bar_test_indices)} words")
    if len(bar_test_indices) == 0:
        raise ValueError("No shared vocabulary after intersection. Lower BAR_MIN_WORD_INSTANCES and rerun the cell.")

    print("\nPrecomputing split similarity matrices...")
    bar_val_splits = [bar_prepare_split(data, bar_test_indices) for data in bar_val_data_list]
    bar_test_splits = [bar_prepare_split(data, bar_test_indices) for data in bar_test_data_list]
    pooled_val_split = bar_pool_val_data(bar_val_splits)
    print(f"Pooled val: {pooled_val_split['similarity'].shape[0]} instances across {len(pooled_val_split['rows_by_word'])} words")

    pooled_test_counts = bar_pool_test_counts(bar_test_instances_list)
    pooled_pseudo_test = bar_pseudo_instances(pooled_test_counts)
    bar_moses_freq = build_moses_freq(bar_vocab_words, bar_subtlex_freq)
    bar_sherlock_freq = build_sherlock_freq(bar_vocab_words, pooled_pseudo_test)
    bar_universal_core_freq = build_universal_core_freq(bar_vocab_words, bar_subtlex_freq)

    subtlex_ref = evaluate_reference_multirun("SUBTLEX", bar_subtlex_freq, bar_subtlex_freq, pooled_val_split, bar_test_splits, bar_test_indices, BAR_MAX_VOCAB)
    subtlex_ref["moses_ovmi"] = compute_moses_ovmi(bar_subtlex_freq.sum())
    subtlex_ref["willett_ovmi"] = compute_willett_ovmi(bar_subtlex_freq.sum())
    print(f"  Moses et al. (SUBTLEX ref): {subtlex_ref['moses_ovmi']:.4f}")
    print(f"  Willett et al. (SUBTLEX ref): {subtlex_ref['willett_ovmi']:.4f}")

    moses_ref = evaluate_reference_multirun("Moses", bar_moses_freq, bar_subtlex_freq, pooled_val_split, bar_test_splits, bar_test_indices, BAR_MAX_VOCAB)
    moses_ref["moses_ovmi"] = moses_invasive_ovmi_under_moses_ref()
    moses_ref["willett_ovmi"] = willett_ovmi_under_moses_ref()
    print(f"  Moses et al. (Moses ref): {moses_ref['moses_ovmi']:.4f}")
    print(f"  Willett et al. (Moses ref): {moses_ref['willett_ovmi']:.4f}")

    sherlock_ref = evaluate_reference_multirun("Sherlock", bar_sherlock_freq, bar_subtlex_freq, pooled_val_split, bar_test_splits, bar_test_indices, BAR_MAX_VOCAB)
    sherlock_ref["moses_ovmi"] = moses_invasive_ovmi_under_sherlock_ref(pooled_pseudo_test)
    sherlock_ref["willett_ovmi"] = willett_ovmi_under_sherlock_ref(pooled_pseudo_test)
    print(f"  Moses et al. (Sherlock ref): {sherlock_ref['moses_ovmi']:.4f}")
    print(f"  Willett et al. (Sherlock ref): {sherlock_ref['willett_ovmi']:.4f}")

    universal_core_ref = evaluate_reference_multirun("Universal core", bar_universal_core_freq, bar_subtlex_freq, pooled_val_split, bar_test_splits, bar_test_indices, BAR_MAX_VOCAB)
    universal_core_ref["moses_ovmi"] = moses_invasive_ovmi_under_universal_core_ref()
    universal_core_ref["willett_ovmi"] = willett_ovmi_under_universal_core_ref()
    print(f"  Moses et al. (Universal core ref): {universal_core_ref['moses_ovmi']:.4f}")
    print(f"  Willett et al. (Universal core ref): {universal_core_ref['willett_ovmi']:.4f}")

    _save_panel_cache(
        BAR_CACHE_PATH,
        bar_run_ids,
        {
            "subtlex_ref": subtlex_ref,
            "moses_ref": moses_ref,
            "sherlock_ref": sherlock_ref,
            "universal_core_ref": universal_core_ref,
        },
        bar_vocab_words,
        {
            "subtlex": bar_subtlex_freq,
            "moses": bar_moses_freq,
            "sherlock": bar_sherlock_freq,
            "universal_core": bar_universal_core_freq,
        },
    )
    print(f"\nSaved cache: {BAR_CACHE_PATH}")

bar_wordcloud_freqs = {
    "subtlex": build_wordcloud_freq_map(bar_vocab_words, bar_subtlex_freq, top_n=BAR_WORDCLOUD_TOP_N),
    "moses": build_wordcloud_freq_map(bar_vocab_words, bar_moses_freq, top_n=BAR_WORDCLOUD_TOP_N),
    "sherlock": build_wordcloud_freq_map(bar_vocab_words, bar_sherlock_freq, top_n=BAR_WORDCLOUD_TOP_N),
    "universal_core": build_wordcloud_freq_map(bar_vocab_words, bar_universal_core_freq, top_n=BAR_WORDCLOUD_TOP_N),
}

print("\n===== Summary =====")
for name, panel in [
    ("SUBTLEX reference", subtlex_ref),
    ("Moses reference", moses_ref),
    ("Sherlock reference", sherlock_ref),
    ("Universal core reference", universal_core_ref),
]:
    print(f"{name}:")
    print(f"  Moses et al.: OV-MI = {panel['moses_ovmi']:.4f}")
    print(f"  Willett et al.: OV-MI = {panel['willett_ovmi']:.4f}")
    print(f"  Greedy (V={panel['greedy_peak_V']}): OV-MI = {panel['test_greedy_mean']:.4f} +/- {panel['test_greedy_se']:.4f}")
    print(f"  Top-k (V={panel['topk_peak_V']}): OV-MI = {panel['test_topk_mean']:.4f} +/- {panel['test_topk_se']:.4f}")

fig = plot_bars_multirun(subtlex_ref, moses_ref, sherlock_ref, universal_core_ref, bar_wordcloud_freqs, BAR_OUTPUT_PATH)
plt.show()



## Greedy vocabulary selection with OVMI improves information transfer

In [ ]:
def group_instances(words: np.ndarray, embeddings: np.ndarray) -> dict[str, np.ndarray]:
    grouped = defaultdict(list)
    for word, embedding in zip(words, embeddings):
        grouped[str(word)].append(embedding)
    return {word: np.stack(values) for word, values in grouped.items()}


def split_frequency(vocab_words: np.ndarray, instances_by_word: dict[str, np.ndarray]) -> np.ndarray:
    word_to_index = {word: i for i, word in enumerate(vocab_words)}
    frequency = np.zeros(len(vocab_words), dtype=np.float64)
    for word, instances in instances_by_word.items():
        if word in word_to_index:
            frequency[word_to_index[word]] = len(instances)
    return frequency


def prepare_split(data: dict[str, np.ndarray], test_indices: np.ndarray) -> dict[str, object]:
    instances_by_word = group_instances(data["target_word"], data["pred_embedding"])
    vocab_words = data["vocab_words"]

    chunks = []
    labels = []
    rows_by_word = {}
    row_start = 0
    for local_index, vocab_index in enumerate(test_indices):
        word_instances = instances_by_word[vocab_words[vocab_index]]
        row_stop = row_start + len(word_instances)
        rows_by_word[local_index] = np.arange(row_start, row_stop)
        chunks.append(word_instances)
        labels.extend([local_index] * len(word_instances))
        row_start = row_stop

    all_instances = np.concatenate(chunks).astype(np.float64)
    retrieval = data["vocab_embeddings"][test_indices].astype(np.float64)
    with np.errstate(all="ignore"):
        similarity = normalize_rows(all_instances) @ normalize_rows(retrieval).T

    return {
        "similarity": similarity,
        "rows_by_word": rows_by_word,
        "labels": np.array(labels, dtype=np.int64),
        "instances_by_word": instances_by_word,
    }


def ovmi_for_subset(
    selected,
    split: dict[str, object],
    reference_frequency: np.ndarray,
    test_indices: np.ndarray,
) -> float:
    selected = np.array(selected, dtype=np.int64)
    if len(selected) <= 1:
        return 0.0

    selected_frequency = reference_frequency[test_indices[selected]]
    if selected_frequency.sum() <= 0:
        return 0.0

    accuracies = subset_accuracies(
        selected,
        split["similarity"],
        split["rows_by_word"],
        split["labels"],
    )
    coverage = selected_frequency.sum() / reference_frequency.sum()
    word_probabilities = selected_frequency / selected_frequency.sum()
    return coverage * in_vocab_mi(float(accuracies.mean()), word_probabilities)


def trace_from_order(order, split, reference_frequency, test_indices):
    selected = []
    trace = []
    for local_index in order:
        selected.append(int(local_index))
        trace.append(ovmi_for_subset(selected, split, reference_frequency, test_indices))
    return np.array(trace, dtype=np.float64)


def greedy_order_on_validation(val_split, reference_frequency, test_indices, max_vocab=None, min_examples: int = 1):
    eligible = [
        local_index
        for local_index, vocab_index in enumerate(test_indices)
        if reference_frequency[vocab_index] > 0
        and len(val_split["rows_by_word"].get(local_index, [])) >= min_examples
    ]
    n_steps = min(len(eligible), max_vocab) if max_vocab is not None else len(eligible)

    selected = []
    trace = []
    for _ in range(n_steps):
        remaining = [i for i in eligible if i not in selected]
        scores = [
            ovmi_for_subset(selected + [candidate], val_split, reference_frequency, test_indices)
            for candidate in remaining
        ]
        best_candidate = remaining[int(np.argmax(scores))]
        selected.append(best_candidate)
        trace.append(max(scores))

    return selected, np.array(trace, dtype=np.float64)

In [ ]:
def compute_greedy_run(seed_dir: Path, max_vocab: int = 110, min_word_instances: int = 5, seed: int = 42):
    val_data = load_prediction_run(seed_dir, split="val")
    full_test_data = load_prediction_run(seed_dir, split="test")
    test_data = full_test_data
    vocab_words = test_data["vocab_words"]
    word_to_index = {word: i for i, word in enumerate(vocab_words)}

    val_instances = group_instances(val_data["target_word"], val_data["pred_embedding"])
    test_instances = group_instances(test_data["target_word"], test_data["pred_embedding"])
    shared_words = [
        word
        for word in val_instances
        if word in test_instances
        and word in word_to_index
    ]
    test_indices = np.array(sorted(word_to_index[word] for word in shared_words), dtype=np.int64)

    val_split = prepare_split(val_data, test_indices)
    test_split = prepare_split(test_data, test_indices)
    subtlex_frequency = subtlex_frequencies_for_vocab(vocab_words)
    val_frequency = split_frequency(vocab_words, val_instances)
    test_frequency = split_frequency(vocab_words, test_instances)

    subtlex_greedy_order, val_subtlex_greedy_trace = greedy_order_on_validation(
        val_split,
        subtlex_frequency,
        test_indices,
        max_vocab=max_vocab,
        min_examples=min_word_instances,
    )
    val_greedy_order, val_greedy_trace = greedy_order_on_validation(
        val_split,
        val_frequency,
        test_indices,
        max_vocab=max_vocab,
        min_examples=min_word_instances,
    )
    peak_v = int(np.nanargmax(val_subtlex_greedy_trace)) + 1

    subtlex_order = np.argsort(subtlex_frequency[test_indices])[::-1][:max_vocab].tolist()
    val_frequency_order = np.argsort(val_frequency[test_indices])[::-1][:max_vocab].tolist()
    full_val_accuracy = subset_accuracies(
        np.arange(len(test_indices), dtype=np.int64),
        val_split["similarity"],
        val_split["rows_by_word"],
        val_split["labels"],
    )
    validation_eligible = [
        local_index
        for local_index in range(len(test_indices))
        if len(val_split["rows_by_word"].get(local_index, [])) >= min_word_instances
    ]
    best_accuracy_order = sorted(
        validation_eligible,
        key=lambda i: (-float(full_val_accuracy[i]), int(i)),
    )[:max_vocab]
    best_accuracy_x_frequency_order = sorted(
        [i for i in validation_eligible if val_frequency[test_indices[i]] > 0],
        key=lambda i: (-float(full_val_accuracy[i] * val_frequency[test_indices[i]]), int(i)),
    )[:max_vocab]
    best_accuracy_x_subtlex_frequency_order = sorted(
        [i for i in validation_eligible if subtlex_frequency[test_indices[i]] > 0],
        key=lambda i: (-float(full_val_accuracy[i] * subtlex_frequency[test_indices[i]]), int(i)),
    )[:max_vocab]
    random_order = np.random.default_rng(seed).permutation(len(test_indices))[:max_vocab].tolist()

    result = {
        "seed": seed_dir.name,
        "test_indices": test_indices,
        "test_words": np.array([vocab_words[i] for i in test_indices]),
        "subtlex_frequency": subtlex_frequency,
        "val_frequency": val_frequency,
        "test_frequency": test_frequency,
        "subtlex_greedy_order": np.array(subtlex_greedy_order, dtype=np.int64),
        "val_greedy_order": np.array(val_greedy_order, dtype=np.int64),
        "test_freq_order": np.array(subtlex_order, dtype=np.int64),
        "test_valfreq_order": np.array(val_frequency_order, dtype=np.int64),
        "test_maxpc_order": np.array(best_accuracy_order, dtype=np.int64),
        "test_maxpcfreq_order": np.array(best_accuracy_x_frequency_order, dtype=np.int64),
        "test_maxpcsubtlexfreq_order": np.array(best_accuracy_x_subtlex_frequency_order, dtype=np.int64),
        "test_rand_order": np.array(random_order, dtype=np.int64),
        "val_subtlex_greedy_trace": val_subtlex_greedy_trace,
        "val_greedy_trace": val_greedy_trace,
        "test_subtlex_greedy_trace": trace_from_order(subtlex_greedy_order, test_split, subtlex_frequency, test_indices),
        "test_greedy_trace": trace_from_order(val_greedy_order, test_split, subtlex_frequency, test_indices),
        "test_freq_trace": trace_from_order(subtlex_order, test_split, subtlex_frequency, test_indices),
        "test_valfreq_trace": trace_from_order(val_frequency_order, test_split, subtlex_frequency, test_indices),
        "test_maxpc_trace": trace_from_order(best_accuracy_order, test_split, subtlex_frequency, test_indices),
        "test_maxpcfreq_trace": trace_from_order(best_accuracy_x_frequency_order, test_split, subtlex_frequency, test_indices),
        "test_maxpcsubtlexfreq_trace": trace_from_order(best_accuracy_x_subtlex_frequency_order, test_split, subtlex_frequency, test_indices),
        "test_rand_trace": trace_from_order(random_order, test_split, subtlex_frequency, test_indices),
        "testp_subtlex_greedy_trace": trace_from_order(subtlex_greedy_order, test_split, test_frequency, test_indices),
        "testp_greedy_trace": trace_from_order(val_greedy_order, test_split, test_frequency, test_indices),
        "testp_freq_trace": trace_from_order(subtlex_order, test_split, test_frequency, test_indices),
        "testp_valfreq_trace": trace_from_order(val_frequency_order, test_split, test_frequency, test_indices),
        "testp_maxpc_trace": trace_from_order(best_accuracy_order, test_split, test_frequency, test_indices),
        "testp_maxpcfreq_trace": trace_from_order(best_accuracy_x_frequency_order, test_split, test_frequency, test_indices),
        "testp_rand_trace": trace_from_order(random_order, test_split, test_frequency, test_indices),
        "peak_v": peak_v,
    }

    for name, order in [
        ("subtlex_greedy", subtlex_greedy_order),
        ("greedy", val_greedy_order),
        ("freq", subtlex_order),
        ("valfreq", val_frequency_order),
        ("maxpc", best_accuracy_order),
        ("maxpcfreq", best_accuracy_x_frequency_order),
        ("maxpcsubtlexfreq", best_accuracy_x_subtlex_frequency_order),
    ]:
        result[f"test_{name}_peak_accs"] = subset_accuracies(
            np.array(order[:peak_v], dtype=np.int64),
            test_split["similarity"],
            test_split["rows_by_word"],
            test_split["labels"],
        )

    return result


GREEDY_MAX_VOCAB = 110
GREEDY_MIN_WORD_INSTANCES = 5

greedy_run_results = []
greedy_peak_rows = []
for seed_dir in seed_dirs:
    result = compute_greedy_run(
        seed_dir,
        max_vocab=GREEDY_MAX_VOCAB,
        min_word_instances=GREEDY_MIN_WORD_INSTANCES,
    )
    greedy_run_results.append(result)
    for label, key in [
        ("val greedy, p=SUBTLEX", "val_subtlex_greedy_trace"),
        ("val greedy, p=validation", "val_greedy_trace"),
        ("test greedy, p=SUBTLEX", "test_subtlex_greedy_trace"),
        ("test greedy, p=validation", "test_greedy_trace"),
        ("test SUBTLEX-frequency", "test_freq_trace"),
        ("test val-frequency", "test_valfreq_trace"),
        ("test best accuracy", "test_maxpc_trace"),
        ("test best accuracy x frequency", "test_maxpcfreq_trace"),
    ]:
        peak_index = int(np.nanargmax(result[key]))
        greedy_peak_rows.append({
            "seed": result["seed"],
            "trace": label,
            "peak_ovmi": result[key][peak_index],
            "peak_vocab_size": peak_index + 1,
            "val_selected_peak_v": result["peak_v"],
        })

pd.DataFrame(greedy_peak_rows)

In [ ]:
GREEDY_COLORS = {
    "greedy": "#e377c2",
    "subtlex_greedy": "#1f77b4",
    "subtlex": "#2ca02c",
    "valfreq": "#ff7f0e",
    "maxpc": "#9467bd",
    "maxpcfreq": "#17becf",
    "random": "#7f7f7f",
}


def plot_line_with_optional_se(axis, x, mean, se, color, label, n_runs, linewidth=1.5, alpha=0.18):
    axis.plot(x, mean, "-", color=color, linewidth=linewidth, label=label)
    if n_runs > 1:
        axis.fill_between(x, mean - se, mean + se, color=color, alpha=alpha, linewidth=0)


def style_arrow_axes(axis):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    arrowprops = {
        "arrowstyle": "-|>",
        "color": "black",
        "linewidth": 0.8,
        "mutation_scale": 8,
        "shrinkA": 0,
        "shrinkB": 0,
    }
    axis.annotate("", xy=(1.02, 0), xytext=(0.97, 0), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrowprops, clip_on=False)
    axis.annotate("", xy=(0, 1.04), xytext=(0, 0.96), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrowprops, clip_on=False)


def stack_greedy_trace(key: str) -> np.ndarray:
    traces = [result[key] for result in greedy_run_results]
    max_length = max(len(trace) for trace in traces)
    stacked = np.full((len(traces), max_length), np.nan)
    for row, trace in enumerate(traces):
        stacked[row, : len(trace)] = trace
    return stacked


greedy_summary = {}
for key in [
    "test_subtlex_greedy_trace",
    "test_greedy_trace",
    "test_freq_trace",
    "test_valfreq_trace",
    "test_maxpc_trace",
    "test_maxpcfreq_trace",
    "test_rand_trace",
    "testp_subtlex_greedy_trace",
    "testp_greedy_trace",
    "testp_freq_trace",
    "testp_valfreq_trace",
    "testp_maxpc_trace",
    "testp_maxpcfreq_trace",
    "testp_rand_trace",
]:
    stacked = stack_greedy_trace(key)
    n = np.sum(~np.isnan(stacked), axis=0)
    greedy_summary[key] = {
        "mean": np.nanmean(stacked, axis=0),
        "se": np.nanstd(stacked, axis=0, ddof=1) / np.sqrt(n),
    }


def annotate_trace_end(axis, values, label, color, x_values=None, alpha=0.75):
    finite = np.where(np.isfinite(values))[0]
    if len(finite) == 0:
        return
    index = int(finite[-1])
    x = x_values[index] if x_values is not None else index + 1
    axis.annotate(label, (x, values[index]), xytext=(0, 2), textcoords="offset points", ha="right", va="bottom", fontsize=8, color=color, alpha=alpha)


def pick_annotated_words(points, n=12, seed=0):
    unique = {word: frequency for word, frequency, _accuracy, _color in points if frequency > 0}
    words = list(unique)
    if not words:
        return set()
    log_frequency = np.log([unique[word] for word in words])
    edges = np.linspace(log_frequency.min(), log_frequency.max(), n + 1)
    rng = np.random.default_rng(seed)
    chosen = set()
    for i in range(n):
        if i == n - 1:
            in_bin = (log_frequency >= edges[i]) & (log_frequency <= edges[i + 1])
        else:
            in_bin = (log_frequency >= edges[i]) & (log_frequency < edges[i + 1])
        indices = np.where(in_bin)[0]
        if len(indices):
            chosen.add(words[int(rng.choice(indices))])
    return chosen


TRACE_SPECS = [
    ("subtlex_greedy_trace", GREEDY_COLORS["subtlex_greedy"], "Greedy OVMI (p=SUBTLEX)"),
    ("greedy_trace", GREEDY_COLORS["greedy"], "Greedy OVMI (p=validation)"),
    ("freq_trace", GREEDY_COLORS["subtlex"], "Most frequent (SUBTLEX)"),
    ("valfreq_trace", GREEDY_COLORS["valfreq"], "Most frequent (validation)"),
    ("maxpc_trace", GREEDY_COLORS["maxpc"], "Best accuracy"),
    ("maxpcfreq_trace", GREEDY_COLORS["maxpcfreq"], "Best accuracy x frequency"),
]
SCATTER_SPECS = [
    ("test_freq_order", "test_freq_peak_accs", GREEDY_COLORS["subtlex"]),
    ("test_valfreq_order", "test_valfreq_peak_accs", GREEDY_COLORS["valfreq"]),
    ("test_maxpc_order", "test_maxpc_peak_accs", GREEDY_COLORS["maxpc"]),
    ("test_maxpcfreq_order", "test_maxpcfreq_peak_accs", GREEDY_COLORS["maxpcfreq"]),
    ("subtlex_greedy_order", "test_subtlex_greedy_peak_accs", GREEDY_COLORS["subtlex_greedy"]),
    ("val_greedy_order", "test_greedy_peak_accs", GREEDY_COLORS["greedy"]),
]

from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("retina")

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def plot_greedy_summary(trace_prefix: str, random_key: str, scatter_frequency_key: str, scatter_xlabel: str):
    sizes = np.arange(1, len(greedy_summary[f"{trace_prefix}subtlex_greedy_trace"]["mean"]) + 1)
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(8, 3))

    for suffix, color, label in TRACE_SPECS:
        key = f"{trace_prefix}{suffix}"
        plot_line_with_optional_se(ax_left, sizes, greedy_summary[key]["mean"], greedy_summary[key]["se"], color, label, len(greedy_run_results))
        peak_index = int(np.nanargmax(greedy_summary[key]["mean"]))
        ax_left.plot(peak_index + 1, greedy_summary[key]["mean"][peak_index], "*", color=color, markersize=10, markeredgecolor="black", markeredgewidth=0.5, zorder=5)

    plot_line_with_optional_se(ax_left, sizes, greedy_summary[random_key]["mean"], greedy_summary[random_key]["se"], GREEDY_COLORS["random"], "_nolegend_", len(greedy_run_results), linewidth=1.0, alpha=0.12)
    annotate_trace_end(ax_left, greedy_summary[random_key]["mean"], "random", GREEDY_COLORS["random"], x_values=sizes, alpha=0.7)

    ax_left.set_xlabel("Vocabulary size")
    ax_left.set_ylabel("bits / word attempt")
    ax_left.set_ylim(bottom=0)
    ax_left.grid(axis="y", alpha=0.3)
    ax_left.grid(axis="x", alpha=0.3)
    style_arrow_axes(ax_left)

    reference_run = greedy_run_results[0]
    peak_v = int(reference_run["peak_v"])
    test_indices = reference_run["test_indices"]
    test_words = reference_run["test_words"]
    scatter_frequency = reference_run[scatter_frequency_key]

    points = []
    for order_key, accuracy_key, color in SCATTER_SPECS:
        local_indices = np.asarray(reference_run[order_key][:peak_v], dtype=np.int64)
        x_frequency = scatter_frequency[test_indices[local_indices]].astype(float)
        accuracies = reference_run[accuracy_key]
        ax_right.scatter(x_frequency, accuracies, s=12, color=color, alpha=0.55, edgecolors="none")
        for local_index, frequency, accuracy in zip(local_indices, x_frequency, accuracies):
            points.append((str(test_words[local_index]).strip().lower(), float(frequency), float(accuracy), color))

    annotated_words = pick_annotated_words(points)
    for word, frequency, accuracy, color in points:
        if word in annotated_words:
            ax_right.annotate(word, (frequency, accuracy), xytext=(4, 4), textcoords="offset points", fontsize=8, color=color)

    ax_right.set_xscale("log")
    ax_right.set_xlabel(scatter_xlabel)
    ax_right.set_ylabel(f"Per-word accuracy (V={peak_v})")
    ax_right.set_ylim(0, 1)
    ax_right.grid(alpha=0.3)
    style_arrow_axes(ax_right)

    handles, legend_labels = ax_left.get_legend_handles_labels()
    legend_columns = int(np.ceil(len(handles) / 2))
    fig.legend(handles, legend_labels, loc="upper center", ncol=legend_columns, bbox_to_anchor=(0.5, 1.0), frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.86])


    if trace_prefix.startswith("testp"):
        fig.savefig("greedy_ovmi_libri.pdf", dpi=600)

    plt.show()


plot_greedy_summary("test_", "test_rand_trace", "subtlex_frequency", "SUBTLEX frequency")
plot_greedy_summary("testp_", "testp_rand_trace", "test_frequency", "LibriBrain test frequency")

## Downstream bar plots

In [ ]:
ensure_package("jiwer")
ensure_package("sentence-transformers", "sentence_transformers")

from collections import Counter, defaultdict
import jiwer
from matplotlib.patches import Patch
from sentence_transformers import SentenceTransformer
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("retina")
except ImportError:
    pass


BAR_SENTENCE_TEST_SENTENCE_LIMIT = 30
BAR_SBERT_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
BAR_GREEDY_TRACE_PREFIX = "test_"

if globals().get("SBERT_MODEL_NAME") == BAR_SBERT_MODEL_NAME and "sbert_model" in globals():
    bar_sbert_model = sbert_model
else:
    bar_sbert_model = SentenceTransformer(BAR_SBERT_MODEL_NAME)

BAR_TRACE_ORDER_SPECS = [
    ("subtlex_greedy", "subtlex_greedy_trace", "subtlex_greedy_order", GREEDY_COLORS["subtlex_greedy"], "Greedy OVMI (p=SUBTLEX)"),
    ("greedy", "greedy_trace", "val_greedy_order", GREEDY_COLORS["greedy"], "Greedy OVMI (p=validation)"),
    ("freq", "freq_trace", "test_freq_order", GREEDY_COLORS["subtlex"], "Most frequent (SUBTLEX)"),
    ("valfreq", "valfreq_trace", "test_valfreq_order", GREEDY_COLORS["valfreq"], "Most frequent (validation)"),
    ("maxpc", "maxpc_trace", "test_maxpc_order", GREEDY_COLORS["maxpc"], "Best accuracy"),
    ("maxpcfreq", "maxpcfreq_trace", "test_maxpcfreq_order", GREEDY_COLORS["maxpcfreq"], "Best accuracy x frequency"),
]
BAR_HATCHES = {
    "subtlex_greedy": "///",
    "greedy": "///",
}
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 1.2,
    "hatch.linewidth": 1.1,
})
BAR_AXIS_LABEL_SIZE = 13
BAR_TICK_LABEL_SIZE = 11
BAR_LEGEND_SIZE = 10
BAR_VALUE_LABEL_SIZE = 8.0


def limit_bar_test_sentences(test_data: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    if BAR_SENTENCE_TEST_SENTENCE_LIMIT is None:
        return test_data
    sentence_keys = np.array([
        f"{subject}:{sentence_idx}"
        for subject, sentence_idx in zip(test_data["subject"], test_data["sentence_idx_of_word"])
    ])
    selected = set(np.unique(sentence_keys)[:BAR_SENTENCE_TEST_SENTENCE_LIMIT])
    keep = np.array([key in selected for key in sentence_keys])
    return {
        key: value[keep] if isinstance(value, np.ndarray) and len(value) == len(keep) else value
        for key, value in test_data.items()
    }


def bar_sentence_text(tokens: list[str]) -> str:
    return " ".join(str(token).lower().strip() for token in tokens if str(token).strip())


def normalize_bar_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)


def nearest_bar_vocab_words(pred_embeddings: np.ndarray, vocab_embeddings_subset: np.ndarray) -> np.ndarray:
    similarity = normalize_bar_rows(pred_embeddings.astype(np.float64)) @ normalize_bar_rows(vocab_embeddings_subset.astype(np.float64)).T
    return similarity.argmax(axis=1)


def decode_bar_sentences(test_data: dict[str, np.ndarray], vocab_global_indices: np.ndarray) -> tuple[list[str], list[str]]:
    vocab_words = test_data["vocab_words"]
    vocab_strings = np.array([str(vocab_words[index]) for index in vocab_global_indices])
    vocab_set = set(vocab_strings.tolist())

    target_words = np.array([str(word) for word in test_data["target_word"]])
    in_vocab = np.array([word in vocab_set for word in target_words])
    predicted_words = np.array([""] * len(target_words), dtype=object)

    if in_vocab.any():
        predicted_local = nearest_bar_vocab_words(
            test_data["pred_embedding"][in_vocab],
            test_data["vocab_embeddings"][vocab_global_indices],
        )
        predicted_words[in_vocab] = vocab_strings[predicted_local]

    sentence_groups = defaultdict(lambda: ([], []))
    for row, word in enumerate(target_words):
        key = (str(test_data["subject"][row]), int(test_data["sentence_idx_of_word"][row]))
        target_tokens, hypothesis_tokens = sentence_groups[key]
        target_tokens.append(word)
        if in_vocab[row]:
            hypothesis_tokens.append(str(predicted_words[row]))

    refs, hyps = [], []
    for target_tokens, hypothesis_tokens in sentence_groups.values():
        if hypothesis_tokens:
            refs.append(bar_sentence_text(target_tokens))
            hyps.append(bar_sentence_text(hypothesis_tokens))
    return refs, hyps


BAR_SBERT_CACHE = {}


def bar_sbert_embeddings(texts: list[str]) -> np.ndarray:
    missing = [text for text in dict.fromkeys(texts) if text not in BAR_SBERT_CACHE]
    if missing:
        embeddings = bar_sbert_model.encode(
            missing,
            convert_to_numpy=True,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        for text, embedding in zip(missing, embeddings):
            BAR_SBERT_CACHE[text] = np.asarray(embedding, dtype=np.float32)
    return np.stack([BAR_SBERT_CACHE[text] for text in texts])


def score_bar_vocabulary(test_data: dict[str, np.ndarray], vocab_global_indices: np.ndarray) -> dict[str, float]:
    refs, hyps = decode_bar_sentences(test_data, vocab_global_indices)
    if not refs:
        return {"wer": np.nan, "sbert_cos": np.nan}
    ref_embeddings = bar_sbert_embeddings(refs)
    hyp_embeddings = bar_sbert_embeddings(hyps)
    return {
        "wer": float(jiwer.wer(refs, hyps)),
        "sbert_cos": float(np.mean(np.sum(ref_embeddings * hyp_embeddings, axis=1))),
    }


BAR_COVERAGE_TARGET = 0.50


def test_token_counts_for_vocab(test_data: dict[str, np.ndarray], test_indices: np.ndarray) -> tuple[np.ndarray, int]:
    counts = Counter(test_data["target_word"].astype(str).tolist())
    token_counts = np.array(
        [counts.get(str(test_data["vocab_words"][global_index]), 0) for global_index in test_indices],
        dtype=np.float64,
    )
    return token_counts, len(test_data["target_word"])


def pick_vocab_size_for_test_coverage(order: np.ndarray, token_counts: np.ndarray, total_tokens: int, target: float) -> tuple[int, float]:
    if len(order) == 0 or total_tokens == 0:
        return 0, 0.0
    coverage = np.cumsum(token_counts[order]) / total_tokens
    index = int(np.searchsorted(coverage, target, side="left"))
    index = min(index, len(order) - 1)
    return index + 1, float(coverage[index])


def pick_vocab_size_for_subtlex_coverage(order: np.ndarray, subtlex_frequency: np.ndarray, test_indices: np.ndarray, target: float) -> tuple[int, float]:
    weights = np.asarray(subtlex_frequency[test_indices], dtype=np.float64)
    positive_total = float(np.sum(weights[weights > 0]))
    if len(order) == 0 or positive_total == 0:
        return 0, 0.0
    coverage = np.cumsum(weights[order]) / positive_total
    index = int(np.searchsorted(coverage, target, side="left"))
    index = min(index, len(order) - 1)
    return index + 1, float(coverage[index])


bar_rows = []
for result in greedy_run_results:
    test_data = limit_bar_test_sentences(load_prediction_run(PREDICTIONS_DIR / result["seed"], split="test"))
    test_indices = result["test_indices"]
    token_counts, total_tokens = test_token_counts_for_vocab(test_data, test_indices)

    for method_key, trace_suffix, order_key, color, label in BAR_TRACE_ORDER_SPECS:
        order = np.asarray(result[order_key], dtype=np.int64)
        for coverage_kind, picker in [
            ("SUBTLEX", lambda: pick_vocab_size_for_subtlex_coverage(order, result["subtlex_frequency"], test_indices, BAR_COVERAGE_TARGET)),
            ("test set", lambda: pick_vocab_size_for_test_coverage(order, token_counts, total_tokens, BAR_COVERAGE_TARGET)),
        ]:
            vocab_size, achieved_coverage = picker()
            scores = score_bar_vocabulary(test_data, test_indices[order[:vocab_size]])
            bar_rows.append({
                "seed": result["seed"],
                "method_key": method_key,
                "method": label,
                "coverage_kind": coverage_kind,
                "coverage_target": BAR_COVERAGE_TARGET,
                "achieved_coverage": achieved_coverage,
                "color": color,
                "vocab_size": vocab_size,
                **scores,
            })

bar_metrics = pd.DataFrame(bar_rows)


def mean_and_se_for_bar(values: pd.Series) -> tuple[float, float]:
    values = values.dropna().astype(float)
    if len(values) == 0:
        return np.nan, 0.0
    if len(values) == 1:
        return float(values.iloc[0]), 0.0
    return float(values.mean()), float(values.std(ddof=1) / np.sqrt(len(values)))


def style_bar_axis(axis) -> None:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_linewidth(1.2)
    axis.spines["bottom"].set_linewidth(1.2)
    axis.tick_params(axis="both", colors="black", width=1.2, labelsize=BAR_TICK_LABEL_SIZE)
    axis.set_axisbelow(True)
    axis.grid(axis="y", alpha=0.22, zorder=0)
    axis.axhline(0, color="black", lw=1.0)
    arrowprops = {
        "arrowstyle": "-|>",
        "color": "black",
        "linewidth": 1.2,
        "mutation_scale": 10,
        "shrinkA": 0,
        "shrinkB": 0,
    }
    axis.annotate("", xy=(0, 1.04), xytext=(0, 0.96), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrowprops, clip_on=False)


def plot_downstream_coverage_bars(frame: pd.DataFrame, coverage_kind: str, coverage_target: float = BAR_COVERAGE_TARGET) -> None:
    target_frame = frame[
        (frame["coverage_kind"] == coverage_kind)
        & np.isclose(frame["coverage_target"], coverage_target)
    ]
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    x = np.arange(len(BAR_TRACE_ORDER_SPECS))
    colors = [color for *_rest, color, _label in BAR_TRACE_ORDER_SPECS]
    legend_handles = [
        Patch(facecolor=color, edgecolor="black", hatch=BAR_HATCHES.get(method_key, ""), label=label)
        for method_key, *_middle, color, label in BAR_TRACE_ORDER_SPECS
    ]

    for axis, metric, ylabel in [
        (axes[0], "sbert_cos", "SBERT Cosine Similarity"),
        (axes[1], "wer", "WER (lower is better)"),
    ]:
        means, errors = [], []
        for method_key, *_ in BAR_TRACE_ORDER_SPECS:
            values = target_frame.loc[target_frame["method_key"] == method_key, metric]
            mean, se = mean_and_se_for_bar(values)
            means.append(mean)
            errors.append(se)

        bars = axis.bar(
            x,
            means,
            yerr=errors,
            color=colors,
            width=0.62,
            alpha=0.9,
            edgecolor="black",
            linewidth=0.8,
            zorder=3,
            error_kw={"elinewidth": 1.5, "ecolor": "black", "capsize": 4.0, "capthick": 1.5},
        )
        for bar, (method_key, *_rest) in zip(bars, BAR_TRACE_ORDER_SPECS):
            bar.set_hatch(BAR_HATCHES.get(method_key, ""))
        for xi, value, error in zip(x, means, errors):
            if np.isfinite(value):
                axis.annotate(
                    f"{value:.3f}",
                    (xi, value + error),
                    textcoords="offset points",
                    xytext=(0, 3),
                    ha="center",
                    fontsize=BAR_VALUE_LABEL_SIZE,
                    fontweight="bold",
                    color="black",
                )

        axis.set_xticks([])
        axis.set_xlim(-0.6, len(BAR_TRACE_ORDER_SPECS) - 0.4)
        axis.set_ylabel(ylabel, fontsize=BAR_AXIS_LABEL_SIZE, color="black")
        top = max((value + error) for value, error in zip(means, errors) if np.isfinite(value))
        axis.set_ylim(0, top * 1.25 if top > 0 else 1.0)
        style_bar_axis(axis)

    axes[0].set_ylim(0.1, 0.4)
    axes[1].set_ylim(0.84, 0.89)

    legend_columns = int(np.ceil(len(legend_handles) / 2))
    fig.legend(
        legend_handles,
        [handle.get_label() for handle in legend_handles],
        loc="upper center",
        ncol=legend_columns,
        bbox_to_anchor=(0.5, 1.0),
        frameon=False,
        fontsize=BAR_LEGEND_SIZE,
        handlelength=1.5,
        columnspacing=1.2,
    )
    # fig.suptitle(f"{coverage_target:.0%} coverage on {coverage_kind}", y=0.83, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.85])
    # fig.tight_layout()
    plt.show()

    fig.savefig("downstream.pdf", dpi=600)


display(bar_metrics.groupby(["coverage_kind", "coverage_target", "method"], sort=False)[["achieved_coverage", "vocab_size", "wer", "sbert_cos"]].agg(["mean", "sem"]))
# plot_downstream_coverage_bars(bar_metrics, "SUBTLEX")
plot_downstream_coverage_bars(bar_metrics, "test set")
